In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

### Running the eval



In [3]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""

    prompt = f"""
    Please solve the following task:

    {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [4]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [5]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [7]:
import json
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [8]:
print(json.dumps(results, indent=2))


[
  {
    "output": "# Python Function to Extract AWS Region from S3 Bucket URL\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_aws_region_from_s3_url(s3_url: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Supports multiple S3 URL formats:\n    - Virtual-hosted-style: s3://my-bucket.us-west-2.amazonaws.com/key\n    - Path-style: s3://s3.us-west-2.amazonaws.com/my-bucket/key\n    - Legacy: s3://my-bucket.s3.amazonaws.com/key (returns 'us-east-1')\n    \n    Args:\n        s3_url (str): The S3 bucket URL\n        \n    Returns:\n        str: The AWS region code (e.g., 'us-west-2')\n        \n    Raises:\n        ValueError: If the URL format is invalid or region cannot be extracted\n    \"\"\"\n    if not s3_url:\n        raise ValueError(\"URL cannot be empty\")\n    \n    # Remove 's3://' prefix if present\n    url = s3_url.replace('s3://', '')\n    \n    # Extract the hostname part\n    hostname = url.split('/')[0]\n  